# Phase 1. EDA & 시즌 분석

**프로젝트:** 카카오톡 선물하기 CRM 분석 포트폴리오  
**분석 기간:** 2023-01-01 ~ 2024-12-31  
**BigQuery:** `ds-ysy.kakao_gift`

## 분석 목표
1. 데이터 품질 검증 (NULL, 중복, FK 무결성)
2. 월별 GMV 트렌드 (MoM)
3. 시즌 이벤트 YoY 비교 (2023 vs 2024)
4. 카테고리 믹스 분석 + HHI 집중도
5. 진입 경로 및 상품 발견 방법 분석
6. 선물 수락률 & 캠페인 차단율

## 0. 환경 설정

In [3]:
# Colab에서 BigQuery 인증하는 방법
from google.colab import auth
auth.authenticate_user()  # ← 이 줄 추가! 브라우저 팝업으로 구글 계정 로그인

from google.cloud import bigquery
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# 한글 폰트 (Colab은 한글 폰트가 없어서 설치 필요)
!pip install -q koreanize-matplotlib
import koreanize_matplotlib

PROJECT = 'ds-ysy'
DATASET = 'kakao_gift'
client  = bigquery.Client(project=PROJECT)

def bq(sql):
    return client.query(sql).to_dataframe()

print('환경 설정 완료')

환경 설정 완료


In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from google.cloud import bigquery

# 한글 폰트
import koreanize_matplotlib

# BigQuery 클라이언트
PROJECT = 'ds-ysy'
DATASET = 'kakao_gift'
client  = bigquery.Client(project=PROJECT)

def bq(sql: str) -> pd.DataFrame:
    """BigQuery 쿼리 실행 헬퍼"""
    return client.query(sql).to_dataframe()

# 시각화 기본 설정
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False
PALETTE = ['#FEE500', '#3C1E1E', '#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4']

print('환경 설정 완료')

환경 설정 완료


---
## 1. 데이터 품질 검증

In [5]:
# 1-1. 테이블별 행 수 확인
tables = ['users', 'orders', 'gift_receipts', 'campaigns', 'campaign_logs']
counts = {}
for t in tables:
    cnt = bq(f'SELECT COUNT(*) AS cnt FROM `{PROJECT}.{DATASET}.{t}`')['cnt'][0]
    counts[t] = cnt
    print(f'{t:20s}: {cnt:>10,}행')

users               :     50,000행
orders              :    200,032행
gift_receipts       :    200,032행
campaigns           :         88행
campaign_logs       :    317,893행


In [6]:
# 1-2. orders NULL 비율 확인
null_check = bq("""
SELECT
  COUNTIF(order_id IS NULL)             / COUNT(*) AS order_id_null,
  COUNTIF(sender_user_id IS NULL)       / COUNT(*) AS sender_null,
  COUNTIF(total_amount_krw IS NULL)     / COUNT(*) AS amount_null,
  COUNTIF(gift_occasion IS NULL)        / COUNT(*) AS occasion_null,
  COUNTIF(created_at IS NULL)           / COUNT(*) AS created_at_null
FROM `ds-ysy.kakao_gift.orders`
""")
print('orders NULL 비율:')
display(null_check.T.rename(columns={0: 'null_rate'}))

orders NULL 비율:


,null_rate
order_id_null,0.0
sender_null,0.0
amount_null,0.0
occasion_null,0.0
created_at_null,0.0


In [7]:
# 1-3. order_status / receipt_status 분포 확인
order_status = bq("""
SELECT order_status, COUNT(*) AS cnt,
  ROUND(COUNT(*) / SUM(COUNT(*)) OVER() * 100, 1) AS pct
FROM `ds-ysy.kakao_gift.orders`
GROUP BY 1 ORDER BY 2 DESC
""")

receipt_status = bq("""
SELECT receipt_status, COUNT(*) AS cnt,
  ROUND(COUNT(*) / SUM(COUNT(*)) OVER() * 100, 1) AS pct
FROM `ds-ysy.kakao_gift.gift_receipts`
GROUP BY 1 ORDER BY 2 DESC
""")

print('order_status 분포:')
display(order_status)
print('\nreceipt_status 분포:')
display(receipt_status)

order_status 분포:


,order_status,cnt,pct
0,accepted,163997,82.0
1,expired,16028,8.0
2,pending_accepted,14028,7.0
3,refunded,5979,3.0



receipt_status 분포:


,receipt_status,cnt,pct
0,accepted,164023,82.0
1,expired,15975,8.0
2,pending,14020,7.0
3,option_changed,6014,3.0


In [8]:
# 1-4. FK 무결성: orders.sender_user_id → users.user_id 불일치 건수
fk_check = bq("""
SELECT COUNT(*) AS orphan_orders
FROM `ds-ysy.kakao_gift.orders` o
LEFT JOIN `ds-ysy.kakao_gift.users` u ON o.sender_user_id = u.user_id
WHERE u.user_id IS NULL
""")
print(f"FK 불일치 주문: {fk_check['orphan_orders'][0]:,}건 (0이면 정상)")

FK 불일치 주문: 0건 (0이면 정상)


---
## 2. 월별 GMV 트렌드 (MoM)

In [ ]:
monthly_gmv = bq("""
SELECT
  FORMAT_DATE('%Y-%m', DATE(created_at))  AS year_month,
  EXTRACT(YEAR FROM created_at)           AS year,
  EXTRACT(MONTH FROM created_at)          AS month,
  SUM(total_amount_krw)                   AS gmv,
  COUNT(order_id)                         AS order_cnt,
  ROUND(AVG(total_amount_krw))            AS aov
FROM `ds-ysy.kakao_gift.orders`
WHERE order_status != 'refunded'
GROUP BY 1, 2, 3
ORDER BY 1
""")

monthly_gmv['gmv_billion'] = monthly_gmv['gmv'] / 1e8  # 억원 단위
monthly_gmv.head()

In [ ]:
# 시즌 이벤트 주요 날짜 (수직선 표시용)
SEASON_EVENTS = {
    '발렌타인(2/14)': '2023-02', '어버이날(5/8)':   '2023-05',
    '빼빼로(11/11)':  '2023-11', '발렌타인(2/14)_24':'2024-02',
    '어버이날(5/8)_24':'2024-05', '빼빼로(11/11)_24': '2024-11',
}

fig, ax1 = plt.subplots(figsize=(14, 5))

# GMV 막대
colors = [PALETTE[0] if y == 2024 else PALETTE[1]
          for y in monthly_gmv['year']]
bars = ax1.bar(monthly_gmv['year_month'], monthly_gmv['gmv_billion'],
               color=colors, alpha=0.85, label='GMV (억원)')
ax1.set_ylabel('GMV (억원)', fontsize=11)
ax1.set_xlabel('')
ax1.tick_params(axis='x', rotation=45)

# AOV 선 (보조 축)
ax2 = ax1.twinx()
ax2.plot(monthly_gmv['year_month'], monthly_gmv['aov'],
         color='#FF6B6B', marker='o', linewidth=2, markersize=4, label='AOV (원)')
ax2.set_ylabel('AOV (원)', fontsize=11, color='#FF6B6B')
ax2.tick_params(axis='y', labelcolor='#FF6B6B')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

# 범례
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor=PALETTE[0], label='2024 GMV'),
    Patch(facecolor=PALETTE[1], label='2023 GMV'),
    plt.Line2D([0], [0], color='#FF6B6B', marker='o', label='AOV'),
]
ax1.legend(handles=legend_elements, loc='upper left')
plt.title('월별 GMV 트렌드 & AOV (2023~2024)', fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('01_monthly_gmv_trend.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# MoM 성장률
monthly_gmv['gmv_mom'] = monthly_gmv['gmv'].pct_change() * 100

# 연간 합계
annual = monthly_gmv.groupby('year').agg(
    total_gmv=('gmv','sum'),
    total_orders=('order_cnt','sum'),
    avg_aov=('aov','mean')
).reset_index()
annual['yoy_gmv'] = annual['total_gmv'].pct_change() * 100
print('연간 GMV 요약:')
display(annual)

---
## 3. 시즌 이벤트 YoY 비교

In [ ]:
season_yoy = bq("""
WITH season_orders AS (
  SELECT
    gift_occasion,
    EXTRACT(YEAR FROM created_at)  AS year,
    SUM(total_amount_krw)          AS gmv,
    COUNT(order_id)                AS order_cnt,
    ROUND(AVG(total_amount_krw))   AS aov
  FROM `ds-ysy.kakao_gift.orders`
  WHERE order_status != 'refunded'
    AND occasion_category = 'special'
    AND gift_occasion != 'day33'
  GROUP BY 1, 2
)
SELECT
  gift_occasion,
  MAX(IF(year=2023, gmv, NULL))       AS gmv_2023,
  MAX(IF(year=2024, gmv, NULL))       AS gmv_2024,
  MAX(IF(year=2023, order_cnt, NULL)) AS orders_2023,
  MAX(IF(year=2024, order_cnt, NULL)) AS orders_2024,
  ROUND(
    (MAX(IF(year=2024, gmv, NULL)) - MAX(IF(year=2023, gmv, NULL)))
    / MAX(IF(year=2023, gmv, NULL)) * 100, 1
  ) AS yoy_pct
FROM season_orders
GROUP BY 1
ORDER BY gmv_2024 DESC
""")

display(season_yoy)

In [ ]:
# Normalized Index: 비시즌 평균 GMV = 100 기준
nonseason_avg = bq("""
SELECT AVG(daily_gmv) AS base
FROM (
  SELECT DATE(created_at) AS dt, SUM(total_amount_krw) AS daily_gmv
  FROM `ds-ysy.kakao_gift.orders`
  WHERE order_status != 'refunded'
    AND occasion_category = 'daily'
  GROUP BY 1
)
""")['base'][0]

season_yoy['index_2023'] = season_yoy['gmv_2023'] / nonseason_avg * 100
season_yoy['index_2024'] = season_yoy['gmv_2024'] / nonseason_avg * 100

# ── 시각화: 3개 차트 (GMV / 주문건수 / YoY 성장률) ──────────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

x = np.arange(len(season_yoy))
w = 0.35

# ① GMV 절대값
axes[0].bar(x - w/2, season_yoy['gmv_2023'] / 1e8, w,
            label='2023', color=PALETTE[1], alpha=0.85)
axes[0].bar(x + w/2, season_yoy['gmv_2024'] / 1e8, w,
            label='2024', color=PALETTE[0], alpha=0.85)
axes[0].set_xticks(x)
axes[0].set_xticklabels(season_yoy['gift_occasion'], rotation=35, ha='right', fontsize=9)
axes[0].set_ylabel('GMV (억원)')
axes[0].set_title('시즌 이벤트별 GMV', fontweight='bold')
axes[0].legend()

# ② 주문건수
axes[1].bar(x - w/2, season_yoy['orders_2023'] / 1e3, w,
            label='2023', color=PALETTE[1], alpha=0.85)
axes[1].bar(x + w/2, season_yoy['orders_2024'] / 1e3, w,
            label='2024', color=PALETTE[0], alpha=0.85)
axes[1].set_xticks(x)
axes[1].set_xticklabels(season_yoy['gift_occasion'], rotation=35, ha='right', fontsize=9)
axes[1].set_ylabel('주문건수 (천 건)')
axes[1].set_title('시즌 이벤트별 주문건수', fontweight='bold')
axes[1].legend()

# ③ YoY 성장률
colors_yoy = ['#FF6B6B' if v < 0 else '#4ECDC4' for v in season_yoy['yoy_pct']]
axes[2].bar(x, season_yoy['yoy_pct'], color=colors_yoy, alpha=0.85)
axes[2].set_xticks(x)
axes[2].set_xticklabels(season_yoy['gift_occasion'], rotation=35, ha='right', fontsize=9)
axes[2].axhline(0, color='gray', linewidth=0.8, linestyle='--')
axes[2].set_ylabel('YoY 성장률 (%)')
axes[2].set_title('시즌 이벤트별 YoY 성장률', fontweight='bold')

for ax in axes:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.suptitle('시즌 이벤트 YoY 비교 (GMV · 주문건수 · 성장률)', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('02_season_yoy.png', dpi=150, bbox_inches='tight')
plt.show()

# ── AOV 비교 테이블 (GMV / 주문건수 → AOV 역산) ─────────────────────
season_yoy['aov_2023'] = (season_yoy['gmv_2023'] / season_yoy['orders_2023']).round(0).astype(int)
season_yoy['aov_2024'] = (season_yoy['gmv_2024'] / season_yoy['orders_2024']).round(0).astype(int)
print('\n시즌 이벤트별 AOV (평균 주문 단가):')
display(season_yoy[['gift_occasion','orders_2023','orders_2024','aov_2023','aov_2024','yoy_pct']])

---
## 4. 카테고리 믹스 분석 + HHI

In [ ]:
category_mix = bq("""
SELECT
  FORMAT_DATE('%Y-Q%Q', DATE(created_at)) AS quarter,
  category_l1,
  SUM(total_amount_krw)                   AS gmv,
  COUNT(order_id)                         AS order_cnt
FROM `ds-ysy.kakao_gift.orders`
WHERE order_status != 'refunded'
GROUP BY 1, 2
ORDER BY 1, 3 DESC
""")

# 분기별 카테고리 비중
category_mix['gmv_share'] = category_mix.groupby('quarter')['gmv'].transform(
    lambda x: x / x.sum() * 100
)

# HHI 계산
hhi_by_quarter = category_mix.groupby('quarter').apply(
    lambda df: ((df['gmv'] / df['gmv'].sum()) ** 2).sum()
).reset_index(name='hhi')

print('분기별 HHI (카테고리 집중도):')
print('  < 0.25: 분산  |  0.25~0.35: 중간  |  > 0.35: 편중')
display(hhi_by_quarter)

In [ ]:
# Stacked Bar (분기별 카테고리 비중)
pivot = category_mix.pivot_table(
    index='quarter', columns='category_l1', values='gmv_share', fill_value=0
)

fig, ax = plt.subplots(figsize=(14, 5))
pivot.plot(kind='bar', stacked=True, ax=ax,
           colormap='tab10', alpha=0.85, width=0.7)
ax.set_xlabel('')
ax.set_ylabel('GMV 비중 (%)')
ax.set_title('분기별 카테고리 GMV 믹스', fontweight='bold', pad=12)
ax.legend(loc='upper right', bbox_to_anchor=(1.15, 1))
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.savefig('03_category_mix.png', dpi=150, bbox_inches='tight')
plt.show()

# HHI 라인
fig2, ax2 = plt.subplots(figsize=(12, 3))
ax2.plot(hhi_by_quarter['quarter'], hhi_by_quarter['hhi'],
         marker='o', color=PALETTE[2], linewidth=2)
ax2.axhline(0.25, color='orange', linestyle='--', alpha=0.7, label='HHI=0.25 기준선')
ax2.set_ylabel('HHI')
ax2.set_title('분기별 카테고리 집중도 (HHI)', fontweight='bold')
ax2.tick_params(axis='x', rotation=30)
ax2.legend()
plt.tight_layout()
plt.savefig('03b_hhi.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. 진입 경로 & 상품 발견 방법 분석

In [ ]:
route_analysis = bq("""
SELECT
  entry_route,
  discovery_method,
  COUNT(order_id)              AS order_cnt,
  ROUND(AVG(total_amount_krw)) AS aov,
  SUM(total_amount_krw)        AS gmv
FROM `ds-ysy.kakao_gift.orders`
WHERE order_status != 'refunded'
GROUP BY 1, 2
ORDER BY gmv DESC
""")

# entry_route별 GMV
entry_gmv = route_analysis.groupby('entry_route').agg(
    gmv=('gmv','sum'), aov=('aov','mean')
).sort_values('gmv', ascending=False)

display(entry_gmv)

In [ ]:
# entry_route × discovery_method AOV 히트맵
pivot_aov = route_analysis.pivot_table(
    index='entry_route', columns='discovery_method',
    values='aov', aggfunc='mean', fill_value=0
)

fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(pivot_aov, annot=True, fmt='.0f', cmap='YlOrRd',
            linewidths=0.5, ax=ax,
            cbar_kws={'label': 'AOV (원)'})
ax.set_title('진입 경로 × 발견 방법별 평균 AOV', fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig('04_route_discovery_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 6. 선물 수락률 & 캠페인 차단율

In [ ]:
# 전체 수락률
acceptance = bq("""
SELECT
  receipt_status,
  COUNT(*) AS cnt,
  ROUND(COUNT(*) / SUM(COUNT(*)) OVER() * 100, 1) AS pct
FROM `ds-ysy.kakao_gift.gift_receipts`
GROUP BY 1 ORDER BY 2 DESC
""")

print('전체 선물 수락률:')
display(acceptance)

In [ ]:
# 시즌 이벤트별 수락률
acceptance_by_season = bq("""
SELECT
  o.gift_occasion,
  COUNT(*)                                              AS total,
  COUNTIF(r.receipt_status = 'accepted')               AS accepted,
  ROUND(COUNTIF(r.receipt_status = 'accepted') / COUNT(*) * 100, 1) AS acceptance_rate
FROM `ds-ysy.kakao_gift.orders` o
JOIN `ds-ysy.kakao_gift.gift_receipts` r ON o.order_id = r.order_id
WHERE o.occasion_category = 'special'
GROUP BY 1
ORDER BY acceptance_rate DESC
""")

display(acceptance_by_season)

In [ ]:
# 캠페인 차단율 (message_type별)
block_rate = bq("""
SELECT
  message_type,
  message_variation_id,
  COUNTIF(event_type = 'send')     AS sent,
  COUNTIF(event_type = 'open')     AS opened,
  COUNTIF(event_type = 'click')    AS clicked,
  COUNTIF(event_type = 'purchase') AS purchased,
  COUNTIF(event_type = 'block')    AS blocked,
  ROUND(COUNTIF(event_type = 'open')    / NULLIF(COUNTIF(event_type = 'send'),0) * 100, 1) AS open_rate,
  ROUND(COUNTIF(event_type = 'click')   / NULLIF(COUNTIF(event_type = 'send'),0) * 100, 1) AS ctr,
  ROUND(COUNTIF(event_type = 'purchase')/ NULLIF(COUNTIF(event_type = 'click'),0) * 100, 1) AS cvr,
  ROUND(COUNTIF(event_type = 'block')   / NULLIF(COUNTIF(event_type = 'send'),0) * 100, 2) AS block_rate
FROM `ds-ysy.kakao_gift.campaign_logs`
GROUP BY 1, 2
ORDER BY 1, 2
""")

print('캠페인 성과 (message_type × A/B variation):')
display(block_rate)

In [ ]:
# 퍼널 시각화 (A vs B)
funnel_ab = block_rate[['message_variation_id', 'sent', 'opened', 'clicked', 'purchased']].copy()
funnel_ab = funnel_ab.groupby('message_variation_id').sum().reset_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i, (_, row) in enumerate(funnel_ab.iterrows()):
    stages = ['sent', 'opened', 'clicked', 'purchased']
    values = [row[s] for s in stages]
    colors_funnel = [PALETTE[0] if row['message_variation_id']=='A' else PALETTE[2]] * 4
    axes[i].barh(stages[::-1], values[::-1], color=colors_funnel, alpha=0.85)
    axes[i].set_title(f"Variation {row['message_variation_id']} 퍼널",
                      fontweight='bold')
    axes[i].set_xlabel('유저 수')
    for j, v in enumerate(values[::-1]):
        axes[i].text(v * 1.01, j, f'{v:,}', va='center', fontsize=9)

plt.suptitle('캠페인 A/B 퍼널 비교', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('05_campaign_funnel.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7. 이상치 탐지 (시즌 피크 vs 비정상)

In [ ]:
daily_gmv = bq("""
SELECT
  DATE(created_at)       AS dt,
  SUM(total_amount_krw)  AS daily_gmv,
  COUNT(order_id)        AS daily_orders,
  gift_occasion
FROM `ds-ysy.kakao_gift.orders`
WHERE order_status != 'refunded'
GROUP BY 1, 4
ORDER BY 1
""")

# 일별 GMV 합산
daily_total = daily_gmv.groupby('dt')['daily_gmv'].sum().reset_index()

# 시즌 더미
season_months = {2, 5, 9, 10, 11, 12, 1}
daily_total['dt'] = pd.to_datetime(daily_total['dt'])
daily_total['is_season'] = daily_total['dt'].dt.month.isin(season_months)

# 비시즌 기반 3σ 상한선
nonseason = daily_total[~daily_total['is_season']]
mu  = nonseason['daily_gmv'].mean()
sig = nonseason['daily_gmv'].std()
upper = mu + 3 * sig

daily_total['is_anomaly'] = (
    (~daily_total['is_season'] & (daily_total['daily_gmv'] > upper)) |
    (daily_total['is_season']  & (daily_total['daily_gmv'] > mu * 8))
)

print(f'비시즌 평균: {mu:,.0f}원 | 3σ 상한: {upper:,.0f}원')
print(f'이상치 감지: {daily_total["is_anomaly"].sum()}일')

In [ ]:
fig, ax = plt.subplots(figsize=(16, 5))

ax.plot(daily_total['dt'], daily_total['daily_gmv'] / 1e6,
        color='#3C1E1E', linewidth=0.8, alpha=0.7, label='일별 GMV')
ax.axhline(upper / 1e6, color='orange', linestyle='--',
           linewidth=1.2, alpha=0.8, label=f'3σ 상한선 ({upper/1e6:.0f}M)')

# 이상치 점 표시
anomalies = daily_total[daily_total['is_anomaly']]
ax.scatter(anomalies['dt'], anomalies['daily_gmv'] / 1e6,
           color='red', s=50, zorder=5, label='이상치')

ax.set_ylabel('GMV (백만원)')
ax.set_title('일별 GMV 트렌드 & 이상치 탐지 (3σ)', fontweight='bold', pad=12)
ax.legend()
plt.tight_layout()
plt.savefig('06_anomaly_detection.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 8. Phase 1 핵심 인사이트 요약

In [ ]:
# 인사이트 자동 출력
yoy = annual['yoy_gmv'].iloc[-1]
pepero_yoy = season_yoy[season_yoy['gift_occasion']=='pepero_day']['yoy_pct'].values
top_entry = entry_gmv.index[0]
top_cat = category_mix.groupby('category_l1')['gmv'].sum().idxmax()

print('=' * 55)
print('Phase 1 핵심 인사이트')
print('=' * 55)
print(f'1. 연간 GMV YoY 성장률:     {yoy:+.1f}%')
if len(pepero_yoy) > 0:
    print(f'2. 빼빼로데이 YoY 성장률:   {pepero_yoy[0]:+.1f}%')
print(f'3. 최다 GMV 진입 경로:      {top_entry}')
print(f'4. 최대 GMV 카테고리:       {top_cat}')
accept_rate = acceptance[acceptance['receipt_status']=='accepted']['pct'].values
if len(accept_rate) > 0:
    print(f'5. 전체 선물 수락률:        {accept_rate[0]:.1f}%')
print('=' * 55)
print()
print('→ Phase 2 RFM 세그멘테이션으로 이동')